# Backends of Deep Agents

Backends of deep agents means the implementation that powers the agents workspace and runtime environment (Virtual File System).

Deep Agents can 
- Create Files
- Read Files
- Edit Files
- Store artifacts
- Maintain states

The backend determines where and how this workspace is implemented.

## Relationship of Backend and workspace

Backend -> Virtual Workspace -> Files,States

## Types of Backend
-   StateBackend
-   StoreBackend
-   FileSystemBackend

### State Backend (Default)
The files are stored in the state of langgraph, the files that are stored in the states in available throughout the graph, and when the agent is shut down the files and the memory is lost.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
GOOGLE_API_KEY=os.getenv('GOOGLE_API_KEY')
GROQ_API_KEY=os.getenv('GROQ_API_KEY')

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
model=ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    api_key=GOOGLE_API_KEY
)

In [3]:
from deepagents import create_deep_agent
from deepagents.backends import StateBackend


# deep agent with statebackend

agent=create_deep_agent(
    model=model,
    backend=StateBackend()
)

In [4]:
response=agent.invoke({
    'messages':[{'role':'user','content':"Create a file at /notes/todo.txt exactly with this content in it :\n"
                 "1.Create video\n"
                 "2.Edit video\n"
                 "3.Upload Video\n"
                 "tell me when you have done it"}]
})

In [5]:
response

{'messages': [HumanMessage(content='Create a file at /notes/todo.txt exactly with this content in it :\n1.Create video\n2.Edit video\n3.Upload Video\ntell me when you have done it', additional_kwargs={}, response_metadata={}, id='6d70e8d6-b542-43d6-9645-47b4886a30c8'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'write_file', 'arguments': '{"file_path": "/notes/todo.txt", "content": "1.Create video\\n2.Edit video\\n3.Upload Video"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019eb62b-4ed5-7e80-9857-856adc039ba9-0', tool_calls=[{'name': 'write_file', 'args': {'file_path': '/notes/todo.txt', 'content': '1.Create video\n2.Edit video\n3.Upload Video'}, 'id': 'fc68bf6d-dd63-497a-b159-42665ed69327', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 5944, 'output_tokens': 41, 'total_tokens': 5985, 'input_token_details': {'cache_re

In [6]:
print('---------------Backend Check ----------------')
files=response.get('files',{})
print(files)

if files:
    for path,content in files.items():
        print(f'Path of the file :{path} \nContent of the file: {content['content']}')


---------------Backend Check ----------------
{'/notes/todo.txt': {'content': '1.Create video\n2.Edit video\n3.Upload Video', 'encoding': 'utf-8', 'created_at': '2026-06-11T10:12:33.809338+00:00', 'modified_at': '2026-06-11T10:12:33.809338+00:00'}}
Path of the file :/notes/todo.txt 
Content of the file: 1.Create video
2.Edit video
3.Upload Video


In [7]:
followup=agent.invoke({'messages':[{'role':'user','content':'Read /notes/todo.txt file back to me'}],'files':response.get('files',{})})

In [8]:
followup

{'messages': [HumanMessage(content='Read /notes/todo.txt file back to me', additional_kwargs={}, response_metadata={}, id='35f46355-b6f9-411f-b86e-0862e4da64c8'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'read_file', 'arguments': '{"file_path": "/notes/todo.txt"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019eb62b-78fb-7751-b5de-95c39977a8e1-0', tool_calls=[{'name': 'read_file', 'args': {'file_path': '/notes/todo.txt'}, 'id': 'badbc97f-8fef-400f-b62b-eb6e7035f7c8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 5915, 'output_tokens': 21, 'total_tokens': 5936, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='     1\t1.Create video\n     2\t2.Edit video\n     3\t3.Upload Video', name='read_file', id='a04b9bac-62b4-4063-ab45-22950ff77728', tool_call_id='badbc97f-8fef-400f-b62b-eb6e7035f7c8'),
  AIMessa

### FileSystemBackend()
Here the agents directly create files in our local storage

In [9]:
from langchain_groq import ChatGroq

model1=ChatGroq(model='openai/gpt-oss-120b',api_key=GROQ_API_KEY)

from deepagents.backends import FilesystemBackend
Root=os.path.abspath(os.getcwd())
agent1=create_deep_agent(
    model=model1,
    backend=FilesystemBackend(root_dir=Root,virtual_mode=False)
)

In [11]:
response = agent1.stream({
    "messages": [{
        "role": "user",
        "content": f"""
Create a file named {Root}/notes/todo1.txt inside the notes folder of the current workspace.

Content:
1. Create a video
2. Edit Video
3. Upload video
"""
    }]
})
for r in response:
    print( r, end="", flush=True)

{'PatchToolCallsMiddleware.before_agent': None}{'model': {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants to create a file at path "d:\\AGENTIC AI\\5.deepagents_learning\\deepagent/notes/todo1.txt". However our filesystem uses Unix-like paths starting with /. So we need to map that. Likely the workspace root is / (maybe). The user says "inside the notes folder of the current workspace." So we need to locate the notes folder. Let\'s list root.', 'tool_calls': [{'id': 'fc_a3cf8e71-7495-4e60-b12b-5b98febc26dd', 'function': {'arguments': '{"path":"/"}', 'name': 'repo_browser.ls'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 108, 'prompt_tokens': 5498, 'total_tokens': 5606, 'completion_time': 0.224406471, 'completion_tokens_details': {'reasoning_tokens': 84}, 'prompt_time': 0.307963861, 'prompt_tokens_details': None, 'queue_time': 0.284710285, 'total_time': 0.532370332}, 'model_name': 'openai/gpt-oss-120b', 'system_fi

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01k73n9pnff5gvg7dxgsaqj1fr` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 196724, Requested 6851. Please try again in 25m44.399999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

### Store Backend 
Creates a deep agent that is backed by langgraph store. If a agent is invoked to write a file on one thread, then after that if we will be able to access the file with another thread. 
It acts as a database, creating a data store.
It is Cross Thread Memory
- files stored in ram , and lost on restart

In [12]:
from deepagents.backends import StoreBackend
from langgraph.store.memory import InMemoryStore
store=InMemoryStore()

agent2=create_deep_agent(
    model=model,
    backend=StoreBackend(),
    store=store,
)

In [13]:
import os
import uuid

thread1={'configurable':{'thread_id':str(uuid.uuid4())}}

response=agent2.invoke({'messages':[{'role':'user','content':"Create a file at /notes/todo.txt exactly with this content in it :\n"
                 "1.Create video\n"
                 "2.Edit video\n"
                 "3.Upload Video\n"
                 "tell me when you have done it"}]},config=thread1)

d:\AGENTIC AI\5.deepagents_learning\.venv\Lib\site-packages\deepagents\backends\store.py:265: LangChainDeprecationWarning: `StoreBackend` without an explicit `namespace` is deprecated and will be removed in deepagents==0.7.0. Pass `namespace=lambda ctx: (...)` to `StoreBackend`.
  return self._get_namespace_legacy()


In [14]:
response

{'messages': [HumanMessage(content='Create a file at /notes/todo.txt exactly with this content in it :\n1.Create video\n2.Edit video\n3.Upload Video\ntell me when you have done it', additional_kwargs={}, response_metadata={}, id='f681f5cc-835e-4c7e-b042-3a5ee78f3d0b'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'write_file', 'arguments': '{"file_path": "/notes/todo.txt", "content": "1.Create video\\n2.Edit video\\n3.Upload Video"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019eb635-473f-7ca3-8b1e-d153362c8930-0', tool_calls=[{'name': 'write_file', 'args': {'file_path': '/notes/todo.txt', 'content': '1.Create video\n2.Edit video\n3.Upload Video'}, 'id': '6bccffff-6770-4573-9faa-3abbae3b343e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 5944, 'output_tokens': 41, 'total_tokens': 5985, 'input_token_details': {'cache_re

In [15]:
thread2={'configurable':{'thread_id':str(uuid.uuid4())}}

response=agent2.invoke({'messages':[{'role':'user','content':"read the file /notes/todo.txt and give me the content"}]},config=thread1)

d:\AGENTIC AI\5.deepagents_learning\.venv\Lib\site-packages\deepagents\backends\store.py:265: LangChainDeprecationWarning: `StoreBackend` without an explicit `namespace` is deprecated and will be removed in deepagents==0.7.0. Pass `namespace=lambda ctx: (...)` to `StoreBackend`.
  return self._get_namespace_legacy()


In [16]:
response

{'messages': [HumanMessage(content='read the file /notes/todo.txt and give me the content', additional_kwargs={}, response_metadata={}, id='c809dadf-9659-497a-8418-1ef82a91bff8'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'read_file', 'arguments': '{"file_path": "/notes/todo.txt"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019eb636-3616-7d31-b008-e7544f661f4e-0', tool_calls=[{'name': 'read_file', 'args': {'file_path': '/notes/todo.txt'}, 'id': '0f2a429d-56d5-4f2b-a077-a6b396d0b4c4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 5918, 'output_tokens': 21, 'total_tokens': 5939, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='     1\t1.Create video\n     2\t2.Edit video\n     3\t3.Upload Video', name='read_file', id='842fb4b5-a1b7-41c5-ac5b-40cf3b126d2d', tool_call_id='0f2a429d-56d5-4f2b-a077-a6b396d0